# Entrenamiento LoRA paso a paso para Mesias

Notebook guiado para inspeccionar datos, validar la configuración y dejar listo un entrenamiento prudente dentro del entorno recomendado del proyecto.

Antes de avanzar, conviene haber leído [README.md](../README.md), [docs/comandos.md](../docs/comandos.md) y [docs/laboratorio.md](../docs/laboratorio.md).

In [ ]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError(
        "No se encontró la raíz de Mesias. Abre este notebook desde cualquier carpeta dentro del proyecto."
    )


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.training.entrenamiento import EntrenamientoConfig
from src.training.preparacion_datos import load_jsonl_examples, prepare_sft_dataset

dataset_path = project_root / "data" / "dataset_entrenamiento_docente.jsonl"
dataset_path

## Carga e inspección del dataset

Revisa tamaño, formato y una muestra antes de entrenar.

Si aquí aparece un problema de estructura o contenido, corrígelo antes de continuar: el objetivo del notebook es hacer visibles esos ajustes de forma controlada.

In [ ]:
examples = load_jsonl_examples(dataset_path)

len(examples), examples[0], examples[-1]

In [ ]:
prepared = prepare_sft_dataset(examples, max_length=256)

summary = {
    "rows": len(prepared),
    "keys": sorted(prepared[0].keys()),
    "input_tokens_first_row": len(prepared[0]["input_ids"]),
}

summary

## Configuración prudente de entrenamiento

Empieza con valores conservadores y súbelos poco a poco.

Este notebook está pensado para un entorno local limpio y con recursos suficientes. Si la máquina va justa, reduce expectativas y valida cada paso antes de lanzar entrenamiento real.

In [ ]:
config = EntrenamientoConfig(
    epochs=1.0,
    train_batch_size=1,
    gradient_accumulation_steps=8,
    dataloader_num_workers=0,
    cpu_threads=4,
    max_length=1024,
    output_dir=project_root / "data" / "checkpoints" / "mesias-qwen25-lora-safe",
)

config

## Preparación del entrenamiento

Primero prepara el flujo sin entrenamiento real.

El lanzador `tools/train_lora_seguro.py` funciona desde la raíz del proyecto y permite validar dataset, parámetros y rutas antes de consumir más recursos.

In [ ]:
tools_command = ".\\venv\\Scripts\\python.exe .\\tools\\train_lora_seguro.py --dataset .\\data\\dataset_entrenamiento_docente.jsonl --prepare-only"
print(tools_command)

## Ejecución real opcional

Quita `--prepare-only` solo después de validar datos, configuración y entorno.

Si algo requiere un ajuste menor antes de entrenar, hazlo aquí: esa fricción forma parte del propósito educativo del proyecto y evita consumir recursos a ciegas.

## Checklist

- El dataset carga bien.
- `prepare_sft_dataset` devuelve claves y longitudes coherentes.
- La configuración sigue siendo prudente para tu máquina y tu entorno.
- La preparación termina sin sorpresas antes de lanzar entrenamiento real.
- Mesias sigue respondiendo después de preparar o entrenar adaptadores.